In [17]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from preprocessing import get_features_and_target
from visualizer import plot_visualizer
import plotly.graph_objects as go
from tabpfn import TabPFNRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

In [18]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [19]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")
sc = StandardScaler()

target_column = "PullTest (N)" 

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)

# Defining Model


In [20]:
#model = 'XGBoost'
model = 'RandomForest'
#model = 'TabPFN'

# Add Physical Columns Pullout and Interfacial_Failure


In [21]:
def compute_interfacial_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = 1 * (np.pi/4) * (4 * np.sqrt(t))**2 * (0.7 * 365) 
     return np.round(f_pull, 1) 

def compute_pullout_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = np.pi * ((4 * np.sqrt(t)) + 2*t)*t*365 
     return np.round(f_pull, 1) 

x_train['Interfacial_Failure'] = compute_interfacial_failure(x_train) 
x_train['Pullout_Failure'] = compute_pullout_failure(x_train) 
x_dev['Interfacial_Failure'] = compute_interfacial_failure(x_dev)
x_dev['Pullout_Failure'] = compute_pullout_failure(x_dev)

x_train_scale = sc.fit_transform(X=x_train)
x_dev_scale = sc.transform(X=x_dev)

In [22]:
x_train.head()

,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm),Interfacial_Failure,Pullout_Failure
0,35,200,0,6.82,1081.47,0.922,0.920,2953.9,5988.6
1,35,1500,0,52.25,2014.73,0.920,0.925,2953.9,5988.6
2,95,200,0,16.57,1321.93,0.912,0.924,2928.2,5902.3
3,95,200,0,41.42,1615.83,0.948,0.939,3014.9,6195.6
4,35,1500,0,63.82,1137.29,0.930,0.937,2986.0,6097.2


In [23]:
y_train.head()

0    2127.7
1    5346.4
2    2350.4
3    2174.8
4    3897.5
Name: PullTest (N), dtype: float64

# Fit Model

In [ ]:
if model == 'TabPFN':

    # Initialize the regressor
    regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
    # To use TabPFN v2:
    # regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
    regressor.fit(x_train_scale, y_train)

    # Predict on the test set
    predictions = regressor.predict(x_dev_scale)

elif model == 'XGBoost':

    # Convert the data into DMatrix format
    dtrain = xgb.DMatrix(x_train_scale, label=y_train)
    dtest = xgb.DMatrix(x_dev_scale, label=y_dev)

    # Set the parameters for the XGBoost model
    params = {
        'objective': 'reg:squarederror',
        'max_depth': 1,
        'eta': 0.57,
        'eval_metric': 'rmse',
    }

    # Train the model
    num_boost_round = 20
    bst = xgb.train(params, dtrain, num_boost_round)

    # Make predictions
    predictions = bst.predict(dtest)

elif model == 'RandomForest':

    # Set the parameters for the Random Forest model
    params = {
            'n_estimators': 19,
            'max_depth': 5,
            'min_samples_split': 6,
            'min_samples_leaf': 3,
            'random_state': 42,
    }

    predictions = RandomForestRegressor(**params).fit(x_train_scale, y_train).predict(x_dev_scale)


# Check Validation Data

In [25]:
# Category array (must be aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

plot_visualizer(
    true_vals=y_dev,
    pred_vals=predictions,
    categories=categories,
    title=f"Validation Samples: True vs Prediction ({model}) by Category - Data-Driven Training with IFPL Features"
)

# Check Validation Loss and R2

In [26]:
# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, predictions)
rmse = root_mean_squared_error(y_dev, predictions)
R2   = r2_score(y_dev, predictions)


print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")


MAE:  125.67
RMSE: 196.50
R2: 0.70


# Cross Validation

In [29]:
cross_df = pd.read_csv("data/train_dev_data.csv")

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

mae_list = []
rmse_list = []
R2_list = []

for fold, (train_index, val_index) in enumerate(skf.split(cross_df["Sample ID"], cross_df["Category"])):
    x_tr, y_tr = get_features_and_target(cross_df.iloc[train_index], target_column) 
    x_val, y_val = get_features_and_target(cross_df.iloc[val_index], target_column)

    x_tr['Interfacial_Failure'] = compute_interfacial_failure(x_tr) 
    x_tr['Pullout_Failure'] = compute_pullout_failure(x_tr) 
    x_val['Interfacial_Failure'] = compute_interfacial_failure(x_val)
    x_val['Pullout_Failure'] = compute_pullout_failure(x_val)

    x_tr_scale = sc.fit_transform(X=x_tr)
    x_val_scale = sc.transform(x_val)

    if model == 'XGBoost':
        dtrain = xgb.DMatrix(x_tr_scale, label=y_tr)
        dval = xgb.DMatrix(x_val_scale, label=y_val)

        params = {
            'objective': 'reg:squarederror',
            'max_depth': 1,
            'eta': 0.57,
            'eval_metric': 'rmse'
        }

        num_boost_round = 20
        bst = xgb.train(params, dtrain, num_boost_round)

        preds = bst.predict(dval)

    elif model == 'TabPFN':
        regressor = TabPFNRegressor()
        regressor.fit(x_tr, y_tr)

        preds = regressor.predict(x_val)
    
    elif model == 'RandomForest':

    # Set the parameters for the Random Forest model
        params = {
            'n_estimators': 19,
            'max_depth': 5,
            'min_samples_split': 6,
            'min_samples_leaf': 3,
            'random_state': 42,
    }

        preds = RandomForestRegressor(**params).fit(x_tr_scale, y_tr).predict(x_val_scale)

    # Categories
    categories= cross_df.iloc[val_index]["Category"].values

    mae  = mean_absolute_error(y_val, preds)
    rmse = root_mean_squared_error(y_val, preds)
    R2   = r2_score(y_val, preds)

    mae_list.append(mae) 
    rmse_list.append(rmse) 
    R2_list.append(R2)

    plot_visualizer(
        true_vals=y_val,
        pred_vals=preds,
        categories=categories,
        title=f"Fold {fold+1}: True vs Prediction ({model}) by Category - Cross-Validation Residual Learning with IFPL Features"
    )

    print(f"\nFold {fold+1}")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R²  :", R2)

mae_mean = np.mean(mae_list) 
rmse_mean = np.mean(rmse_list) 
R2_mean = np.mean(R2_list) 



Fold 1
MAE : 146.54412316846265
RMSE: 212.45683695872475
R²  : 0.6885972822833152



Fold 2
MAE : 142.83471372408098
RMSE: 252.53069718180507
R²  : 0.7080957924102391



Fold 3
MAE : 140.26469353524072
RMSE: 222.09239860483993
R²  : 0.663930749050562


In [30]:
print(f"mean MAE:  {mae_mean:.2f} ± {np.std(mae_list):.2f}")
print(f"mean RMSE: {rmse_mean:.2f} ± {np.std(rmse_list):.2f}")
print(f"mean R²:   {R2_mean:.2f} ± {np.std(R2_list):.2f}")


mean MAE:  143.21 ± 2.58
mean RMSE: 229.03 ± 17.08
mean R²:   0.69 ± 0.02


# Test Baseline Model (mit Explode)

In [31]:
# Load the training and development datasets
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/test_data.csv")
sc = StandardScaler()

target_column = "PullTest (N)" 

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)

In [32]:
#model = 'XGBoost'
model = 'RandomForest'
#model = 'TabPFN'

In [33]:
def compute_interfacial_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = 1 * (np.pi/4) * (4 * np.sqrt(t))**2 * (0.7 * 365) 
     return np.round(f_pull, 1) 

def compute_pullout_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = np.pi * ((4 * np.sqrt(t)) + 2*t)*t*365 
     return np.round(f_pull, 1) 

x_train['Interfacial_Failure'] = compute_interfacial_failure(x_train) 
x_train['Pullout_Failure'] = compute_pullout_failure(x_train) 
x_dev['Interfacial_Failure'] = compute_interfacial_failure(x_dev)
x_dev['Pullout_Failure'] = compute_pullout_failure(x_dev)

x_train_scale = sc.fit_transform(X=x_train)
x_dev_scale = sc.transform(X=x_dev)

In [34]:
if model == 'TabPFN':

    # Initialize the regressor
    regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
    # To use TabPFN v2:
    # regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
    regressor.fit(x_train_scale, y_train)

    # Predict on the test set
    predictions = regressor.predict(x_dev_scale)

elif model == 'XGBoost':

    # Convert the data into DMatrix format
    dtrain = xgb.DMatrix(x_train_scale, label=y_train)
    dtest = xgb.DMatrix(x_dev_scale, label=y_dev)

    # Set the parameters for the XGBoost model
    params = {
        'objective': 'reg:squarederror',
        'max_depth': 1,
        'eta': 0.57,
        'eval_metric': 'rmse',
    }

    # Train the model
    num_boost_round = 20
    bst = xgb.train(params, dtrain, num_boost_round)

    # Make predictions
    predictions = bst.predict(dtest)

elif model == 'RandomForest':

    # Set the parameters for the Random Forest model
    params = {
            'n_estimators': 19,
            'max_depth': 5,
            'min_samples_split': 6,
            'min_samples_leaf': 3,
            'random_state': 42,
    }

    predictions = RandomForestRegressor(**params).fit(x_train_scale, y_train).predict(x_dev_scale)


In [35]:
# Category array (must be aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

plot_visualizer(
    true_vals=y_dev,
    pred_vals=predictions,
    categories=categories,
    title=f"Validation Samples: True vs Prediction ({model}) by Category - Data-Driven Training with IFPL Features"
)

In [36]:
# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, predictions)
rmse = root_mean_squared_error(y_dev, predictions)
R2   = r2_score(y_dev, predictions)


print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")


MAE:  165.33
RMSE: 273.83
R2: 0.73
